In [1]:
import torch
import time
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
from CMAPSSDataset import *
import CMAPSSDataset as cmap
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Usando: cuda
GPU: NVIDIA GeForce RTX 2060
VRAM disponible: 6.4 GB


## GRN

In [2]:
class GatedResidualNetwork(nn.Module):
    """
    Bloque fundamental del TFT.

    Parámetros:
        input_dim   — dimensión de la entrada principal
        hidden_dim  — dimensión del espacio interno de procesamiento
        output_dim  — dimensión de la salida
        dropout     — regularización
        context_dim — dimensión del contexto externo (opcional)
                      se usa cuando el GRN recibe información adicional
                      por ejemplo: variables estáticas condicionando
                      el procesamiento de variables dinámicas
    """
    def __init__(self, input_dim, hidden_dim, output_dim,
                 dropout=0.1, context_dim=None):
        super().__init__()

        # 1: transformación no lineal 
        # Dos capas lineales con ELU entre ellas
        self.linear_1 = nn.Linear(input_dim, hidden_dim)
        self.linear_2 = nn.Linear(hidden_dim, output_dim)

        # 2: contexto externo
        # Si llega contexto, se suma antes de la primera activación
        self.context_projection = nn.Linear(context_dim, hidden_dim,
                                            bias=False) \
                                  if context_dim is not None else None

        # 3: gate
        # Produce dos vectores del mismo tamaño:
        self.gate = nn.Linear(output_dim, output_dim * 2)

        # 4: proyección residual 
        self.residual_projection = nn.Linear(input_dim, output_dim,
                                             bias=False) \
                                   if input_dim != output_dim else None

        # 5: normalización y regularización
        self.layer_norm = nn.LayerNorm(output_dim)
        self.dropout    = nn.Dropout(dropout)

    def forward(self, x, context=None):
        """
        x:       (batch, ..., input_dim)   — entrada principal
        context: (batch, ..., context_dim) — contexto externo (opcional)

        Los '...' pueden ser una dimensión temporal (batch, T, dim)
        o nada (batch, dim) — el GRN funciona igual en ambos casos
        """

        # 1: transformación no lineal 
        h = self.linear_1(x)                    # (batch, ..., hidden_dim)

        # Inyectar contexto si existe
        # El contexto modula la transformación desde el principio
        if context is not None and self.context_projection is not None:
            h = h + self.context_projection(context)

        h = F.elu(h)                            # activación suave
        h = self.linear_2(h)                    # (batch, ..., output_dim)
        h = self.dropout(h)

        # 2: gate
        # gate produce output_dim*2 valores
        gate_input  = self.gate(h)
        values, gates = gate_input.chunk(2, dim=-1)
        # chunk divide el tensor en 2 partes iguales por la última dimensión
        h = values * torch.sigmoid(gates)       # (batch, ..., output_dim)
        # cada dimensión se multiplica por un valor entre 0 y 1
        # el modelo aprende qué dimensiones dejar pasar

        # 3: conexión residual + normalización
        # Proyectar la entrada si las dimensiones no coinciden
        residual = self.residual_projection(x) \
                   if self.residual_projection is not None else x

        out = self.layer_norm(residual + h)     # (batch, ..., output_dim)
        return out

## VSN Vectorizado

In [3]:
class VariableSelectionNetwork(nn.Module):
    def __init__(self, input_dim, num_vars, hidden_dim,
                 dropout=0.1, context_dim=None):
        super().__init__()
        self.num_vars = num_vars
        self.hidden_dim = hidden_dim
        self.input_dim = input_dim

        # Conv1d con groups=num_vars — pesos independientes por variable
        self.var_linear1 = nn.Conv1d(
            num_vars * input_dim, num_vars * hidden_dim,
            kernel_size=1, groups=num_vars
        )
        self.var_linear2 = nn.Conv1d(
            num_vars * hidden_dim, num_vars * hidden_dim,
            kernel_size=1, groups=num_vars
        )
        self.var_gate = nn.Conv1d(
            num_vars * hidden_dim, num_vars * hidden_dim * 2,
            kernel_size=1, groups=num_vars
        )
        self.var_residual = nn.Conv1d(
            num_vars * input_dim, num_vars * hidden_dim,
            kernel_size=1, groups=num_vars, bias=False
        ) if input_dim != hidden_dim else None

        self.var_norm    = nn.LayerNorm(hidden_dim)
        self.var_dropout = nn.Dropout(dropout)

        self.weight_grn = GatedResidualNetwork(
            input_dim   = input_dim * num_vars,
            hidden_dim  = hidden_dim,
            output_dim  = num_vars,
            dropout     = dropout,
            context_dim = context_dim
        )
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, context=None):
        # x: (batch, T, num_vars, input_dim)
        batch, T, V, D = x.shape

        # Reshape para Conv1d
        # Conv1d espera (N, C, L) → usamos L=1, C=V*D
        x_conv = x.reshape(batch * T, V * D, 1)

        # GRN vectorizado por variable 
        h = F.elu(self.var_linear1(x_conv))      # (batch*T, V*hidden, 1)
        h = self.var_linear2(h)
        h = self.var_dropout(h)

        # GLU
        values, gates = self.var_gate(h).chunk(2, dim=1)
        h = values * torch.sigmoid(gates)         # (batch*T, V*hidden, 1)

        # Residual
        res = self.var_residual(x_conv) if self.var_residual else x_conv
        h = h + res

        # LayerNorm — reshape a (batch*T*V, hidden)
        h = h.squeeze(-1).reshape(batch * T * V, self.hidden_dim)
        h = self.var_norm(h)

        # h: (batch*T*V, hidden) → (batch, T, V, hidden)
        h = h.reshape(batch, T, V, self.hidden_dim)

        # Pesos de importancia
        x_concat = x.reshape(batch, T, -1)        # (batch, T, V*input_dim)
        weights  = self.softmax(
            self.weight_grn(x_concat, context)
        )                                          # (batch, T, V)

        # Suma ponderada
        # (batch, T, V, hidden) × (batch, T, V) → (batch, T, hidden)
        combined = torch.einsum("btvh,btv->bth", h, weights)

        return combined, weights

In [4]:
class FeatureProjection(nn.Module):
    """
    Proyecta cada feature escalar a un vector de dimensión input_dim.
    Convierte (batch, T, n_features) → (batch, T, n_features, input_dim)
    """
    def __init__(self, n_features, input_dim):
        super().__init__()
        # Una capa lineal por feature 
        self.projections = nn.ModuleList([
            nn.Linear(1, input_dim)
            for _ in range(n_features)
        ])

    def forward(self, x):
        # x: (batch, T, n_features)
        projected = []
        for i, proj in enumerate(self.projections):
            # x[:, :, i:i+1] → (batch, T, 1) — feature i como vector columna
            projected.append(proj(x[:, :, i:i+1]))  # (batch, T, input_dim)

        # Apilar → (batch, T, n_features, input_dim)
        return torch.stack(projected, dim=2)

## LSTM Encoder

In [5]:
class LSTMEncoder(nn.Module):
    """
    Procesa la secuencia del VSN y produce:
      - sequence_out : todos los estados ocultos → va a la atención
      - context      : estado final comprimido  → inicializa GRNs posteriores

    Parámetros:
        input_dim   — dimensión de entrada (= hidden_dim del VSN)
        hidden_dim  — dimensión del estado oculto del LSTM
        num_layers  — capas apiladas (2 es suficiente para CMAPSS)
        dropout     — dropout entre capas (solo actúa si num_layers > 1)
    """
    def __init__(self, input_dim, hidden_dim, num_layers=2, dropout=0.1):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # LSTM unidireccional con num_layers > 1
        self.lstm = nn.LSTM(
            input_size  = input_dim,
            hidden_size = hidden_dim,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0
        )

        # GRN para comprimir el estado final en un context vector
        self.context_grn = GatedResidualNetwork(
            input_dim  = hidden_dim,
            hidden_dim = hidden_dim,
            output_dim = hidden_dim,
            dropout    = dropout
        )

    def forward(self, x):
        """
        x: (batch, timesteps, input_dim)  ← salida del VSN

        Devuelve:
            sequence_out : (batch, timesteps, hidden_dim)
                           todos los estados ocultos — para la atención
            context      : (batch, hidden_dim)
                           resumen del estado final — para los GRNs
        """
        # El LSTM procesa la secuencia completa de una vez
        sequence_out, (h_n, c_n) = self.lstm(x)
        # sequence_out: (batch, T, hidden_dim)
        # h_n:          (num_layers, batch, hidden_dim)
        # c_n:          (num_layers, batch, hidden_dim)

        # Tomar solo la última capa del estado oculto final
        context = self.context_grn(h_n[-1])
        # context: (batch, hidden_dim)

        return sequence_out, context

## Multi-Headed Attention

In [6]:
class TemporalSelfAttention(nn.Module):
    """
    Multi-head self-attention sobre la secuencia del LSTM encoder.
    Cada timestep atiende a todos los demás timesteps.

    Parámetros:
        hidden_dim  — dimensión de entrada y salida
        num_heads   — número de cabezas de atención en paralelo
                      hidden_dim debe ser divisible por num_heads
        dropout     — dropout sobre los pesos de atención
    """
    def __init__(self, hidden_dim, num_heads=4, dropout=0.1):
        super().__init__()

        assert hidden_dim % num_heads == 0, \
            f"hidden_dim ({hidden_dim}) debe ser divisible por num_heads ({num_heads})"

        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads  # dimensión por cabeza

        # batch_first=True para que acepte (batch, T, dim)
        self.attention = nn.MultiheadAttention(
            embed_dim   = hidden_dim,
            num_heads   = num_heads,
            dropout     = dropout,
            batch_first = True
        )

        # GRN post-atención — decide cuánta información de la atención
        # es útil y cuánta descartar mediante el gate
        self.post_attention_grn = GatedResidualNetwork(
            input_dim  = hidden_dim,
            hidden_dim = hidden_dim,
            output_dim = hidden_dim,
            dropout    = dropout
        )

        # LayerNorm para la conexión residual
        self.layer_norm = nn.LayerNorm(hidden_dim)

    def forward(self, x, mask=None):
        """
        x:    (batch, T, hidden_dim)  ← sequence_out del LSTM encoder
        mask: (T, T) opcional         ← máscara causal si quieres
                                        que t solo vea t' <= t

        Devuelve:
            output:       (batch, T, hidden_dim)
                          cada timestep enriquecido con contexto global
            attn_weights: (batch, T, T)
                          peso que cada timestep da a cada otro
                          directamente interpretable — muestra qué
                          ciclos se "miran" entre sí
        """
        # Self-attention: Q, K, V son todos la misma secuencia x
        # el modelo aprende qué parte de x es útil como Query,
        # Key y Value mediante las proyecciones internas de nn.MultiheadAttention
        attn_out, attn_weights = self.attention(
            query = x,
            key   = x,
            value = x,
            attn_mask = mask,
            need_weights = True,    # devolver los pesos para visualización
            average_attn_weights = False  # mantener pesos por cabeza
        )
        # attn_out:     (batch, T, hidden_dim)
        # attn_weights: (batch, num_heads, T, T)

        # Conexión residual + LayerNorm
        x = self.layer_norm(x + attn_out)

        # GRN post-atención — filtra información no útil
        output = self.post_attention_grn(x)

        return output, attn_weights

## Regression with Quantil Uncertainty

In [7]:
class QuantileRegressionHead(nn.Module):
    """
    Cabeza de regresión con cuantiles para el TFT.

    Predicción de RUL
    con tres cuantiles: p10 (optimista), p50 (mediana), p90 (pesimista).

    Parámetros:
        hidden_dim  — dimensión de entrada (= hidden_dim de la atención)
        quantiles   — lista de cuantiles a predecir
        dropout     — regularización
    """
    def __init__(self, hidden_dim, quantiles=[0.1, 0.5, 0.9], dropout=0.1):
        super().__init__()

        self.quantiles  = quantiles
        self.n_quantiles = len(quantiles)

        # Un GRN + Linear por cuantil para separarlos
        self.heads = nn.ModuleList([
            nn.Sequential(
                GatedResidualNetwork(
                    input_dim = hidden_dim,
                    hidden_dim = hidden_dim,
                    output_dim = hidden_dim,
                    dropout = dropout,
                    context_dim = hidden_dim
                ),
                nn.Linear(hidden_dim,1)
            )
            for _ in self.quantiles
        ])

    def forward(self, attn_output, context):
        """
        attn_output: (batch, T, hidden_dim)  ← salida de la atención
        context:     (batch, hidden_dim)     ← context vector del encoder

        Devuelve:
            quantiles: (batch, n_quantiles)
                       [RUL_p10, RUL_p50, RUL_p90]
        """
        # Usar el ultimo timestep
        x = attn_output[:, -1, :]       # (batch, hidden_dim)

        preds = []
        for head in self.heads:
            grn, linear = head[0], head[1]
            h = grn(x, context) # (batch, hidden_dim)
            preds.append(linear(h)) # (batch, 1)

        return torch.cat(preds, dim=1) # (batch, n_quantiles)


# Función de pérdida: Quantile Loss

def quantile_loss(y_pred, y_true, quantiles=[0.1, 0.5, 0.9]):
    """
    Pérdida de cuantiles (Pinball Loss).

    Para cada cuantil q:
      - si y_true > y_pred: penaliza con q        × error
      - si y_true < y_pred: penaliza con (1-q)    × error

    y_pred: (batch, n_quantiles)
    y_true: (batch,)
    """
    losses = []

    for i, q in enumerate(quantiles):
        pred_q = y_pred[:, i]              # predicción del cuantil q
        error  = y_true - pred_q           # error: positivo si subestimamos

        # Pérdida asimétrica por cuantil
        loss_q = torch.where(
            error >= 0,
            q * error,           # subestimación: penalizar con q
            (q - 1) * error      # sobreestimación: penalizar con (1-q)
        )
        losses.append(loss_q.mean())

    return torch.stack(losses).mean()

## TFT Builder

In [8]:
class TemporalFusionTransformer(nn.Module):
    """
    TFT adaptado para regresión de RUL en CMAPSS.

    Adaptaciones respecto al paper original (Lim et al. 2021):
      - Sin variables conocidas en el futuro (no hay decoder separado)
      - Cabeza de regresión con cuantiles en lugar de forecasting multi-horizonte
      - VSN unificado para variables dinámicas (sensores + config. operacional)
      - Variables estáticas opcionales (dataset ID, condición operacional)

    Parámetros:
        n_features      — número de features de entrada (sensores + derivadas + op)
        hidden_dim      — dimensión interna de todos los bloques
        num_heads       — cabezas de atención (hidden_dim debe ser divisible)
        num_lstm_layers — capas del encoder LSTM
        dropout         — regularización global
        quantiles       — cuantiles a predecir
        static_dim      — dimensión de variables estáticas opcionales
                          si es None no se usan variables estáticas
    """
    def __init__(
        self,
        n_features,
        hidden_dim = 64,
        num_heads = 4,
        num_lstm_layers = 2,
        dropout = 0.1,
        quantiles = [0.1, 0.5, 0.9],
        static_dim = None
    ):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.quantiles = quantiles
        self.use_static = static_dim is not None

        # Bloque 1: proyección de features escalares a vectores
        # cada feature escalar → vector de hidden_dim // 4 dimensiones
        # hidden_dim // 4 es el input_dim del VSN
        self.input_dim = hidden_dim // 4 
        self.feature_projection = FeatureProjection(
            n_features = n_features,
            input_dim  = self.input_dim
        )

        # Bloque 2: Variable Selection Network
        # context_dim = hidden_dim si hay variables estáticas
        # las variables estáticas condicionan qué sensores importan
        self.vsn = VariableSelectionNetwork(
            input_dim   = self.input_dim,
            num_vars    = n_features,
            hidden_dim  = hidden_dim,
            dropout     = dropout,
            context_dim = hidden_dim if self.use_static else None
        )

        # Variables estáticas
        # si se usan, se proyectan al espacio hidden_dim
        # y se pasan como contexto al VSN y al encoder
        if self.use_static:
            self.static_grn = GatedResidualNetwork(
                input_dim  = static_dim,
                hidden_dim = hidden_dim,
                output_dim = hidden_dim,
                dropout    = dropout
            )

        # Bloque 3: LSTM encoder
        self.encoder = LSTMEncoder(
            input_dim   = hidden_dim,
            hidden_dim  = hidden_dim,
            num_layers  = num_lstm_layers,
            dropout     = dropout
        )

        # Bloque 4: atención multi-cabezal temporal
        self.attention = TemporalSelfAttention(
            hidden_dim = hidden_dim,
            num_heads  = num_heads,
            dropout    = dropout
        )

        # GRN entre encoder y atención
        # enriquece la secuencia antes de la atención
        self.pre_attention_grn = GatedResidualNetwork(
            input_dim   = hidden_dim,
            hidden_dim  = hidden_dim,
            output_dim  = hidden_dim,
            dropout     = dropout,
            context_dim = hidden_dim   # condicionado por context del encoder
        )

        # Bloque 5: cabeza de regresión con cuantiles
        self.head = QuantileRegressionHead(
            hidden_dim = hidden_dim,
            quantiles  = quantiles,
            dropout    = dropout
        )

    def forward(self, x, static=None):
        """
        x:      (batch, T, n_features)   ← ventana de ciclos
        static: (batch, static_dim)      ← variables estáticas opcionales

        Devuelve:
            quantiles:    (batch, n_quantiles)  ← [p10, p50, p90]
            vsn_weights:  (batch, T, n_features) ← importancia por sensor
            attn_weights: (batch, heads, T, T)  ← qué ciclos se atienden
        """
        #  1: proyectar features escalares
        x_proj = self.feature_projection(x)
        # (batch, T, n_features) → (batch, T, n_features, input_dim)

        # 2: procesar variables estáticas si existen
        static_context = None
        if self.use_static and static is not None:
            static_context = self.static_grn(static)
            # (batch, static_dim) → (batch, hidden_dim)

            static_context_seq = static_context.unsqueeze(1).expand(
                -1, x.shape[1], -1
            )
            # (batch, hidden_dim) → (batch, T, hidden_dim)
        else:
            static_context_seq = None

        # 3: VSN — selección de variables
        vsn_out, vsn_weights = self.vsn(x_proj, static_context_seq)
        # vsn_out:     (batch, T, hidden_dim)
        # vsn_weights: (batch, T, n_features)

        # 4: LSTM encoder
        sequence_out, context = self.encoder(vsn_out)
        # sequence_out: (batch, T, hidden_dim)
        # context:      (batch, hidden_dim)

        # 5: GRN pre-atención condicionado por context 
        # el context del encoder informa a cada timestep
        # sobre el estado global de la secuencia antes de atender
        context_seq = context.unsqueeze(1).expand(-1, x.shape[1], -1)
        # (batch, hidden_dim) → (batch, T, hidden_dim)

        enriched = self.pre_attention_grn(sequence_out, context_seq)
        # (batch, T, hidden_dim)

        # 6: atención multi-cabezal
        attn_out, attn_weights = self.attention(enriched)
        # attn_out:     (batch, T, hidden_dim)
        # attn_weights: (batch, heads, T, T)

        # 7: cabeza de regresión con cuantiles
        quantiles = self.head(attn_out, context)
        # (batch, n_quantiles)

        return quantiles, vsn_weights, attn_weights

## Preparar Dataset

In [9]:
def preparar_dataset_kmeans(ruta_base,subset,window_size=60):

    config = cmap.CONFIG_DATASETS[subset]
    rul_max = config["rul_max"]
    n_condiciones = config["n_condiciones"]

    COLS = (
        ["motor_id", "ciclo"] +
        [f"op_{i}" for i in range(1, 4)] +
        [f"s{i}"   for i in range(1, 22)]
    )

    train = pd.read_csv(f"{ruta_base}/train_{subset}.txt", sep=r"\s+", header=None, names=COLS)
    test = pd.read_csv(f"{ruta_base}/test_{subset}.txt", sep=r"\s+", header=None, names=COLS)
    rul = pd.read_csv(f"{ruta_base}/RUL_{subset}.txt", sep=r"\s+", header=None, names=["rul_final"])

    train_df = cmap.add_rul_train(train.copy(), rul_max=rul_max)
    test_df = cmap.add_rul_test(test.copy(), rul, rul_max=rul_max)

    # 1. Eliminar constantes (sobre train)
    sensor_cols = [c for c in train_df.columns if c.startswith("s")]
    std = train_df[sensor_cols].std()
    constantes = std[std < 0.01].index.tolist()
    train_df = train_df.drop(columns=constantes)
    test_df = test_df.drop(columns=constantes)
    sensor_cols = [c for c in sensor_cols if c not in constantes]

    # 2. SPLIT POR MOTOR — antes de cualquier fit
    motores = train_df["motor_id"].unique()
    gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
    idx_tr, idx_val = next(gss.split(motores, groups=motores))
    motores_train = motores[idx_tr]
    motores_val = motores[idx_val]

    train_solo = train_df[train_df["motor_id"].isin(motores_train)].copy()
    val_solo = train_df[train_df["motor_id"].isin(motores_val)].copy()

    # 3. KMeans
    if n_condiciones > 1:
        train_solo, km = cmap.identificar_condicion_operacional(train_solo, n_condiciones)
        val_solo["condicion"] = km.predict(val_solo[["op_1","op_2","op_3"]])
        test_df["condicion"] = km.predict(test_df[["op_1","op_2","op_3"]])
    else:
        train_solo["condicion"] = 0
        val_solo["condicion"] = 0
        test_df["condicion"] = 0

    # 4. Features de degradación
    print(f"[{subset}] Calculando features de degradación...")
    deg_cols = [c for c in train_solo.columns if c.startswith("s")]
    train_solo = cmap.añadir_features_degradacion_rapido(train_solo, deg_cols)
    val_solo = cmap.añadir_features_degradacion_rapido(val_solo, deg_cols)
    test_df = cmap.añadir_features_degradacion_rapido(test_df, deg_cols)

    # 5. Normalización 
    feature_cols = [c for c in train_solo.columns if c.startswith("s") or c.startswith("op")]
    train_solo, val_solo, test_df, scalers = cmap.normalizar_por_condicion(
        train_solo, val_solo, test_df, feature_cols, n_condiciones
    )

    # 6. Ventanas 
    X_train, y_train = cmap.crear_ventanas(train_solo, feature_cols, window_size)
    X_val, y_val = cmap.crear_ventanas(val_solo, feature_cols, window_size)
    X_test, y_test = cmap.crear_ventanas_test(test_df, feature_cols, window_size)

    # 7. Variables estáticas (condición operacional para FD002/FD004)
    usar_static = n_condiciones > 1
    if usar_static:
        static_train = cmap.extraer_condicion_por_ventana(train_solo, window_size, n_condiciones)
        static_val = cmap.extraer_condicion_por_ventana(val_solo, window_size, n_condiciones)
        static_test = cmap.extraer_condicion_por_ventana_test(test_df, window_size, n_condiciones)
    else:
        static_train = None
        static_val = None
        static_test = None

    print(f"Motores únicos en train: {len(motores_train)}")
    print(f"Motores únicos en val:   {len(motores_val)}")
    print(f"[{subset}] Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
    print(f"[{subset}] Features: {len(feature_cols)} | Condiciones: {n_condiciones}")

    return (X_train, y_train), (X_val, y_val), (X_test, y_test), scalers, feature_cols, (static_train, static_val, static_test)

## Train TFT

In [10]:
def entrenar_tft(subset, ruta_base="CMAPSSData", window_size=60,
                 epochs=300): # aumentado el número de épocas, habiendo cambiado el quantil loss y creando una cabeza por quantil, incertidumbre epistémica

    batch_size = 384 if subset == "FD004" else 256 if subset == "FD002" else 128
    t0 = time.time()
    # Datos con variables estáticas
    (X_tr, y_tr), (X_val, y_val), (X_te, y_te), _, feature_cols, \
    (static_tr, static_val, static_te) = preparar_dataset_kmeans(
        ruta_base, subset, window_size
    )
    print(f"Preprocesamiento total: {time.time()-t0:.1f}s")
    print(static_tr)
    
    n_condiciones = cmap.CONFIG_DATASETS[subset]["n_condiciones"]
    hidden_dim = cmap.CONFIG_DATASETS[subset]["lstm_units"]

    train_loader = DataLoader(
        CMAPSSDataset(X_tr, y_tr, static_tr),
        batch_size=batch_size, shuffle=True,
        pin_memory=False, num_workers=0 #4 bajar el cuello de botella
    )
    val_loader = DataLoader(
        CMAPSSDataset(X_val, y_val, static_val),
        batch_size=batch_size, shuffle=False,
        pin_memory=False # antes true, bajar el cuello de botella
    )

    # Modelo con static_dim
    # Para FD001/FD003 static_dim=1 (siempre condicion=0, poco útil)
    # Para FD002/FD004 static_dim=6 (6 condiciones operacionales)
    # Solo activar variables estáticas donde aportan información real
    usar_static = n_condiciones > 1
    static_dim  = n_condiciones if usar_static else None
    
    tft = TemporalFusionTransformer(
        n_features = X_tr.shape[2],
        hidden_dim = hidden_dim,
        num_heads = 4,
        num_lstm_layers = 2,
        dropout = 0.15 if subset in ["FD002", "FD004"] else 0.10, 
        quantiles = [0.1, 0.5, 0.9],
        static_dim = static_dim
    ).to(DEVICE)
 
    optimizer = torch.optim.Adam(tft.parameters(), 1e-3, weight_decay=1e-5)
    
    # Schedulers 
    warmup_epochs = 5 if subset == "FD004" else 15

    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return epoch / warmup_epochs
        return 1.0

    warmup_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    if subset == "FD004":
        # CosineAnnealingWarmRestarts — sube el lr periódicamente
        # para escapar de mínimos locales
        
        main_scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=20, T_mult=2, eta_min=1e-7
        )
    else:
        main_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, factor=0.5, patience=15, min_lr=1e-7
        )

    # AMP 
    scaler = torch.amp.GradScaler('cuda')
    
    mejor_rmse, paciencia, mejor_epoch = float("inf"), 0, 0
    historial = {"train_loss": [], "train_rmse": [], "val_rmse": []}

    print(f"AMP habilitado: {torch.cuda.is_available()}")
    print(f"Scheduler: {'CosineAnnealingWarmRestarts' if subset == 'FD004' else 'ReduceLROnPlateau'}")

    t_data, t_forward, t_backward = 0.0, 0.0, 0.0
    
    for epoch in range(1, epochs + 1):

        # Entrenamiento
        tft.train()
        preds_p50, trues = [], []
        train_loss = 0.0

        for batch in train_loader:
            t0 = time.time()
            if usar_static:
                X_b, y_b, s_b = batch
                s_b = s_b.to(DEVICE)
            else:
                X_b, y_b = batch
                s_b = None

            X_b = X_b.to(DEVICE)
            y_b = y_b.to(DEVICE)
            t_data += time.time() - t0

            optimizer.zero_grad()

            t0 = time.time()
            # Forward en float16
            with torch.amp.autocast('cuda'):
                q_pred, _, _ = tft(X_b, static=s_b)
                loss = quantile_loss_ponderada(q_pred, y_b)
            t_forward += time.time() - t0

            t0 = time.time()
            # Backward con scaler
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(tft.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            t_backward += time.time() - t0

            train_loss += loss.item()
            preds_p50.append(q_pred[:, 1].detach().cpu()) # detach añadido por retención innecesaria del grafo computacional
            trues.append(y_b.detach().cpu()) # detach añadido por retención innecesaria del grafo computacional

        train_loss /= len(train_loader)
        preds_p50 = torch.cat(preds_p50)
        trues = torch.cat(trues)
        train_rmse  = torch.sqrt(F.mse_loss(preds_p50, trues)).item()

        # Validación 
        tft.eval()
        preds_p50, trues = [], []

        with torch.no_grad():
            
            for batch in val_loader:
                if usar_static:
                    X_b, y_b, s_b = batch
                    s_b = s_b.to(DEVICE)
                else:
                    X_b, y_b = batch
                    s_b = None
                    
                q_pred, _, _ = tft(X_b.to(DEVICE), static = s_b)
                preds_p50.append(q_pred[:, 1].detach().cpu())  
                trues.append(y_b.detach().cpu()) # detach añadido por retención innecesaria del grafo computacional

        preds_p50 = torch.cat(preds_p50)
        trues = torch.cat(trues)
        val_rmse = torch.sqrt(F.mse_loss(preds_p50, trues)).item()

        # Schedulers
        if epoch < warmup_epochs:
            warmup_scheduler.step()
        else:
            if subset == "FD004":
                main_scheduler.step(epoch - warmup_epochs)
            else:
                main_scheduler.step(val_rmse)

        historial["train_loss"].append(train_loss)
        historial["val_rmse"].append(val_rmse)

        if epoch % 10 == 0:
            print(f"[{subset}] Época {epoch:>3} | "
                  f"train_loss {train_loss:.4f} | "
                  f"train_rmse {train_rmse:.4f} | "
                  f"val_rmse {val_rmse:.2f} | "
                  f"lr {optimizer.param_groups[0]['lr']:.2e} | "
                  f"t_data {t_data:.1f}s | t_fwd {t_forward:.1f}s | t_bwd {t_backward:.1f}s")
            t_data, t_forward, t_backward = 0.0, 0.0, 0.0

        # Early stopping y checkpoint
        if val_rmse < mejor_rmse:
            mejor_rmse, paciencia, mejor_epoch = val_rmse, 0, epoch
            torch.save(tft.state_dict(), f"mejor_tft_{subset}.pt")
        else:
            paciencia += 1
            if paciencia >= 50 if subset == "FD004" else 30:
                print(f"Early stopping en época: {epoch}")
                print(f"Checkpoint en época: {mejor_epoch}")
                break

    tft.load_state_dict(torch.load(f"mejor_tft_{subset}.pt", weights_only=True))
    return tft, historial, (X_te, y_te, static_te)


def quantile_loss_ponderada(q_pred, y_true,quantiles=[0.1, 0.5, 0.9], separacion_min=1):
    losses = []
    for i, q in enumerate(quantiles):
        pred_q = q_pred[:, i]
        error  = y_true - pred_q
        loss_q = torch.where(
            error >= 0,
            q * error,
            (q - 1) * error
        )
        losses.append(loss_q.mean())
    # Penalización por cruce — p10 debe ser menor que p50
    # y p50 menor que p90
    for i in range(len(quantiles) - 1):
        cruce = torch.clamp(
            q_pred[:, i] - q_pred[:, i+1] + separacion_min, min=0
        )
        losses.append(cruce.mean())
    return torch.stack(losses).mean()

In [11]:
def evaluar_tft(tft, X_test, y_test, subset, static_test=None):
    n_condiciones = cmap.CONFIG_DATASETS[subset]["n_condiciones"]
    usar_static = n_condiciones > 1
    tft.eval()

    with torch.no_grad():
        s = torch.tensor(static_test).to(DEVICE) \
            if usar_static and static_test is not None else None
        q_pred, vsn_w, attn_w = tft(
            torch.tensor(X_test).to(DEVICE), static=s
        )

    q_pred = q_pred.cpu().numpy()
    p10 = q_pred[:, 0]
    p50 = q_pred[:, 1]
    p90 = q_pred[:, 2]

    # Métricas sobre p50 — para comparar con CNN-BiLSTM y XGBoost
    diff = p50 - y_test
    score  = np.where(diff < 0,
                      np.exp(-diff / 13.0) - 1,
                      np.exp( diff / 10.0) - 1)
    rmse_v = np.sqrt(np.mean(diff**2))
    mae_v = np.mean(np.abs(diff))

    # Métricas de calibración de los cuantiles
    # p10 debería estar por debajo del real el 10% de las veces
    # p90 debería estar por debajo del real el 90% de las veces
    cobertura_p10 = np.mean(y_test > p10)
    cobertura_p90 = np.mean(y_test > p90)
    banda_media = np.mean(p90 - p10)

    print(f"\n── {subset} ─────────────────────────────")
    print(f"RMSE  (p50): {rmse_v:.2f}")
    print(f"MAE   (p50): {mae_v:.2f}")
    print(f"Score (p50): {score.sum():.1f}")
    print(f"\nCalibración cuantiles:")
    print(f"  Cobertura p10: {cobertura_p10:.2%}  (objetivo 90%)")
    print(f"  Cobertura p90: {cobertura_p90:.2%}  (objetivo 10%)")
    print(f"  Banda media p10-p90: {banda_media:.1f} ciclos")

    return q_pred, vsn_w.cpu().numpy(), attn_w.cpu().numpy()

## Execute TFT

In [21]:
RUTA = "../CMAPSSData"
resultados_tft = {}

for subset in ["FD001", "FD002", "FD003", "FD004"]:
    print(f"\n{'='*50}\nEntrenando TFT — {subset}")
    start = time.time()
    tft, hist, (X_te, y_te, static_te) = entrenar_tft(subset, ruta_base=RUTA)
    end = time.time()
    t = end - start
    print(f"Tiempo entrenando [{subset}]: {t/60:.2f}")
    q_pred, vsn_w, attn_w   = evaluar_tft(tft, X_te, y_te, subset, static_te)
    resultados_tft[subset]  = {
        "tft": tft, "q_pred": q_pred,
        "vsn_weights": vsn_w, "attn_weights": attn_w,
        "y_test": y_te
    }


Entrenando TFT — FD001
[FD001] Calculando features de degradación...
Motores únicos en train: 80
Motores únicos en val:   20
[FD001] Train: (11841, 60, 59) | Val: (2890, 60, 59) | Test: (100, 60, 59)
[FD001] Features: 59 | Condiciones: 1
Preprocesamiento total: 6.5s
None
[FD001] Época  10 | train_loss 12.7132 | train_rmse 53.2832 | val_rmse 49.50 | lr 6.67e-04t_data 0.0s | t_fwd 3.3s | t_bwd 7.6s
[FD001] Época  20 | train_loss 5.9679 | train_rmse 41.0319 | val_rmse 41.01 | lr 9.33e-04t_data 0.0s | t_fwd 3.2s | t_bwd 7.4s
[FD001] Época  30 | train_loss 2.1649 | train_rmse 14.6446 | val_rmse 13.20 | lr 9.33e-04t_data 0.0s | t_fwd 3.3s | t_bwd 9.7s
[FD001] Época  40 | train_loss 1.0745 | train_rmse 8.5905 | val_rmse 12.42 | lr 9.33e-04t_data 0.0s | t_fwd 3.2s | t_bwd 7.6s
[FD001] Época  50 | train_loss 0.6649 | train_rmse 5.2133 | val_rmse 12.98 | lr 4.67e-04t_data 0.1s | t_fwd 3.2s | t_bwd 8.6s
[FD001] Época  60 | train_loss 0.4742 | train_rmse 3.6817 | val_rmse 13.09 | lr 2.33e-04t_dat

KeyboardInterrupt: 

In [12]:
import gc
gc.collect()
torch.cuda.empty_cache()
memoria_libre = torch.cuda.mem_get_info()[0] / 1e9
print(f"Memoria GPU libre: {memoria_libre:.2f} GB")
# Debe ser >= 5.5 GB para continuar

Memoria GPU libre: 5.35 GB


In [13]:
RUTA = "../CMAPSSData"
resultados_tft = {}

for subset in ["FD004"]:
    print(f"\n{'='*50}\nEntrenando TFT — {subset}")
    start = time.time()
    tft, hist, (X_te, y_te, static_te) = entrenar_tft(subset, ruta_base=RUTA)
    end = time.time()
    t = end - start
    print(f"Tiempo entrenando [{subset}]: {t/60:.2f}")
    q_pred, vsn_w, attn_w   = evaluar_tft(tft, X_te, y_te, subset, static_te)
    resultados_tft[subset]  = {
        "tft": tft, "q_pred": q_pred,
        "vsn_weights": vsn_w, "attn_weights": attn_w,
        "y_test": y_te
    }


Entrenando TFT — FD004
[FD004] Calculando features de degradación...
Motores únicos en train: 199
Motores únicos en val:   50
[FD004] Train: (37553, 60, 83) | Val: (9005, 60, 83) | Test: (248, 60, 83)
[FD004] Features: 83 | Condiciones: 6
Preprocesamiento total: 22.0s
[[0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 1.]
 ...
 [0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 1.]]
AMP habilitado: True
Scheduler: CosineAnnealingWarmRestarts
[FD004] Época  10 | train_loss 7.2736 | train_rmse 50.3445 | val_rmse 50.72 | lr 8.54e-04 | t_data 1.7s | t_fwd 24.1s | t_bwd 4497.5s
[FD004] Época  20 | train_loss 2.0850 | train_rmse 16.6208 | val_rmse 21.21 | lr 1.47e-04 | t_data 1.7s | t_fwd 25.7s | t_bwd 4735.2s
[FD004] Época  30 | train_loss 1.8229 | train_rmse 15.5749 | val_rmse 21.13 | lr 9.62e-04 | t_data 1.7s | t_fwd 19.5s | t_bwd 5500.5s
[FD004] Época  40 | train_loss 1.4363 | train_rmse 12.8601 | val_rmse 21.57 | lr 6.91e-04 | t_data 1.8s | t_fwd 20.1s | t_bwd 5090.8s
[F